In [54]:
import pandas as pd

# Load all 5 files
all_countries = pd.read_csv("../data/raw/IDS_ALLCountries_Data.csv", encoding="latin1")
country_meta   = pd.read_csv("../data/raw/IDS_CountryMetaData.csv", encoding="latin1")
series_meta    = pd.read_csv("../data/raw/IDS_SeriesMetaData.csv", encoding="latin1")
footnote_meta  = pd.read_csv("../data/raw/IDS_FootNoteMetaData.csv", encoding="latin1")
country_series = pd.read_csv("../data/raw/Country-Series - Metadata.csv", encoding="latin1")

# Print shape of each so we know size before previewing
print("ALLCountries_Data:", all_countries.shape)
print("CountryMetaData:", country_meta.shape)
print("SeriesMetaData:", series_meta.shape)
print("FootNoteMetaData:", footnote_meta.shape)
print("Country-Series Metadata:", country_series.shape)

ALLCountries_Data: (62983, 39)
CountryMetaData: (134, 30)
SeriesMetaData: (574, 12)
FootNoteMetaData: (2673, 5)
Country-Series Metadata: (375, 4)


In [55]:
print(all_countries.columns.tolist())

['Country Name', 'Country Code', 'Counterpart-Area Name', 'Counterpart-Area Code', 'Series Name', 'Series Code', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032']


In [56]:
all_countries.head()

,Country Name,Country Code,Counterpart-Area Name,Counterpart-Area Code,Series Name,Series Code,2000,2001,2002,2003,...,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032
0,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.DPPG,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.OFFT,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.PRVT,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,World,WLD,Average grant element on new external debt com...,DT.GRE.DPPG,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,World,WLD,Average grant element on new external debt com...,DT.GRE.OFFT,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
year_columns = [str(year) for year in range(2000, 2033)]

df_long = all_countries.melt(
    id_vars=["Country Name", "Country Code", "Counterpart-Area Name", "Counterpart-Area Code", "Series Name", "Series Code"],
    value_vars=year_columns,
    var_name="Year",
    value_name="Value"
)

df_long.shape

(2078439, 8)

In [58]:
df_long.head()

,Country Name,Country Code,Counterpart-Area Name,Counterpart-Area Code,Series Name,Series Code,Year,Value
0,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.DPPG,2000,NaN
1,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.OFFT,2000,NaN
2,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.PRVT,2000,NaN
3,Afghanistan,AFG,World,WLD,Average grant element on new external debt com...,DT.GRE.DPPG,2000,NaN
4,Afghanistan,AFG,World,WLD,Average grant element on new external debt com...,DT.GRE.OFFT,2000,NaN


In [59]:
df_long["Value"].isna().sum()

np.int64(733393)

In [60]:
df_long["Value"].isna().mean() * 100

np.float64(35.28576013055952)

In [61]:
df_long["Year"] = df_long["Year"].astype(int)   # convert Year from string to number
df_long = df_long[(df_long["Year"] >= 2000) & (df_long["Year"] <= 2024)]
df_long.shape

(1574575, 8)

In [62]:
df_long.head(100)

,Country Name,Country Code,Counterpart-Area Name,Counterpart-Area Code,Series Name,Series Code,Year,Value
0,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.DPPG,2000,NaN
1,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.OFFT,2000,NaN
2,Afghanistan,AFG,World,WLD,Average grace period on new external debt comm...,DT.GPA.PRVT,2000,NaN
3,Afghanistan,AFG,World,WLD,Average grant element on new external debt com...,DT.GRE.DPPG,2000,NaN
4,Afghanistan,AFG,World,WLD,Average grant element on new external debt com...,DT.GRE.OFFT,2000,NaN
...,...,...,...,...,...,...,...,...
95,Afghanistan,AFG,World,WLD,"GG, official creditors (INT, current US$)",DT.INT.OFFT.GG.CD,2000,NaN
96,Afghanistan,AFG,World,WLD,"GG, official creditors (NFL, current US$)",DT.NFL.OFFT.GG.CD,2000,NaN
97,Afghanistan,AFG,World,WLD,"GG, official creditors (NTR, current US$)",DT.NTR.OFFT.GG.CD,2000,NaN
98,Afghanistan,AFG,World,WLD,"GG, official creditors (TDS, current US$)",DT.TDS.OFFT.GG.CD,2000,NaN


In [63]:
# Sort so years are in order within each group - required for fill to make sense
df_long = df_long.sort_values(["Country Name", "Series Code", "Year"])

# Fill within each Country + Indicator group only
df_long["Value"] = df_long.groupby(["Country Name", "Series Code"])["Value"].transform(
    lambda x: x.bfill().ffill()
)

df_long.shape

(1574575, 8)

In [64]:
df_long["Value"].isna().sum()

np.int64(87125)

In [65]:
df_long.dtypes

Country Name              object
Country Code              object
Counterpart-Area Name     object
Counterpart-Area Code     object
Series Name               object
Series Code               object
Year                       int64
Value                    float64
dtype: object

In [66]:
df_long.duplicated().sum()

np.int64(50)

In [67]:
df_long[df_long.duplicated(keep=False)].sort_values(["Country Name", "Series Code", "Year"]).head(10)

,Country Name,Country Code,Counterpart-Area Name,Counterpart-Area Code,Series Name,Series Code,Year,Value
62978,NaN,NaN,NaN,NaN,NaN,NaN,2000,NaN
62979,NaN,NaN,NaN,NaN,NaN,NaN,2000,NaN
62980,NaN,NaN,NaN,NaN,NaN,NaN,2000,NaN
125961,NaN,NaN,NaN,NaN,NaN,NaN,2001,NaN
125962,NaN,NaN,NaN,NaN,NaN,NaN,2001,NaN
125963,NaN,NaN,NaN,NaN,NaN,NaN,2001,NaN
188944,NaN,NaN,NaN,NaN,NaN,NaN,2002,NaN
188945,NaN,NaN,NaN,NaN,NaN,NaN,2002,NaN
188946,NaN,NaN,NaN,NaN,NaN,NaN,2002,NaN
251927,NaN,NaN,NaN,NaN,NaN,NaN,2003,NaN


In [68]:
df_long = df_long.dropna(subset=["Country Name", "Series Code"])
df_long.shape

(1574450, 8)

In [69]:
df_long.duplicated().sum()
df_long.shape

(1574450, 8)

In [70]:
df_long.duplicated().sum()

np.int64(0)

In [71]:
df_long = df_long.dropna(subset=["Value"])

In [72]:
df_long["Value"].isna().sum()

np.int64(0)

In [74]:
df_long.head()

,Country Name,Country Code,Counterpart-Area Name,Counterpart-Area Code,Series Name,Series Code,Year,Value
112,Afghanistan,AFG,World,WLD,"Imports of goods, services and primary income ...",BM.GSR.TOTL.CD,2000,3.873397e+09
63095,Afghanistan,AFG,World,WLD,"Imports of goods, services and primary income ...",BM.GSR.TOTL.CD,2001,3.873397e+09
126078,Afghanistan,AFG,World,WLD,"Imports of goods, services and primary income ...",BM.GSR.TOTL.CD,2002,3.873397e+09
189061,Afghanistan,AFG,World,WLD,"Imports of goods, services and primary income ...",BM.GSR.TOTL.CD,2003,3.873397e+09
252044,Afghanistan,AFG,World,WLD,"Imports of goods, services and primary income ...",BM.GSR.TOTL.CD,2004,3.873397e+09


In [75]:
df_long["Country Code"] = df_long["Country Code"].str.strip()
df_long["Series Code"] = df_long["Series Code"].str.strip()

In [21]:
df_long.to_csv("../data/cleaned/All_country_data_cleaned.csv", index=False)

In [22]:
#CountryMeta

In [23]:
print(country_meta.columns.tolist())
country_meta.head()

['Code', 'Long Name', 'Income Group', 'Region', 'Lending category', 'Other groups', 'Currency Unit', 'Latest population census', 'Latest household survey', 'Special Notes', 'National accounts base year', 'National accounts reference year', 'System of National Accounts', 'SNA price valuation', 'PPP survey years', 'Balance of Payments Manual in use', 'External debt Reporting status', 'System of trade', 'Government Accounting concept', 'IMF data dissemination standard', 'Source of most recent Income and expenditure data', 'Vital registration complete', 'Latest agricultural census', 'Latest industrial data', 'Latest trade data', 'Latest water withdrawal data', '2-alpha code', 'WB-2 code', 'Table Name', 'Short Name']


,Code,Long Name,Income Group,Region,Lending category,Other groups,Currency Unit,Latest population census,Latest household survey,Special Notes,...,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data,2-alpha code,WB-2 code,Table Name,Short Name
0,AFG,Islamic State of Afghanistan,Low income,Middle East & North Africa,IDA,HIPC,Afghan afghani,1979,Multiple Indicator Cluster Survey 2022-2023,The reporting period for national accounts dat...,...,"Integrated household survey (IHS), 2016/17",NaN,NaN,NaN,2018.0,2000.0,AF,AF,Afghanistan,Afghanistan
1,ALB,Republic of Albania,Upper middle income,Europe & Central Asia,IBRD,NaN,Albanian lek,2023,Demographic and Health Survey 2017 - 2018,NaN,...,Living Standards Measurement Study Survey (LSM...,Yes,2012,2013.0,2018.0,2006.0,AL,AL,Albania,Albania
2,DZA,People's Democratic Republic of Algeria,Upper middle income,Middle East & North Africa,IBRD,NaN,Algerian dinar,2022,Multiple Indicator Cluster Survey 2018-2019,NaN,...,"Integrated household survey (IHS), 2011",NaN,NaN,2010.0,2017.0,2012.0,DZ,DZ,Algeria,Algeria
3,AGO,People's Republic of Angola,Lower middle income,Sub-Saharan Africa,IBRD,NaN,Angolan kwanza,2014,Demographic and Health Survey 2015/16,The World Bank systematically assesses the app...,...,"Integrated household survey (IHS), 2008/09",NaN,NaN,NaN,2018.0,2005.0,AO,AO,Angola,Angola
4,ARG,Argentine Republic,Upper middle income,Latin America & Caribbean,IBRD,NaN,Argentine peso,2022,Multiple Indicator Cluster Survey 2019-2020,The World Bank systematically assesses the app...,...,"Integrated household survey (IHS), 2016",Yes,2008,2002.0,2018.0,2011.0,AR,AR,Argentina,Argentina


In [24]:
country_meta_clean = country_meta[["Code", "Table Name", "Region", "Income Group", "Lending category"]].copy()
country_meta_clean.columns = ["Country Code", "Country Name", "Region", "Income Group", "Lending Category"]
country_meta_clean.head()

,Country Code,Country Name,Region,Income Group,Lending Category
0,AFG,Afghanistan,Middle East & North Africa,Low income,IDA
1,ALB,Albania,Europe & Central Asia,Upper middle income,IBRD
2,DZA,Algeria,Middle East & North Africa,Upper middle income,IBRD
3,AGO,Angola,Sub-Saharan Africa,Lower middle income,IBRD
4,ARG,Argentina,Latin America & Caribbean,Upper middle income,IBRD


In [25]:
country_meta_clean.isna().sum()

Country Code         0
Country Name         0
Region              14
Income Group        15
Lending Category    14
dtype: int64

In [26]:
country_meta_clean.duplicated().sum()

np.int64(0)

In [27]:
country_meta_clean[country_meta_clean["Region"].isna()]

,Country Code,Country Name,Region,Income Group,Lending Category
32,EAP,East Asia & Pacific (excluding high income),NaN,NaN,NaN
40,ECA,Europe & Central Asia (excluding high income),NaN,NaN,NaN
53,IDX,IDA only,NaN,NaN,NaN
54,IDA,IDA total,NaN,NaN,NaN
66,LAC,Latin America & Caribbean (excluding high income),NaN,NaN,NaN
67,LDC,Least developed countries: UN classification,NaN,NaN,NaN
71,LMY,Low & middle income,NaN,NaN,NaN
72,LIC,Low income,NaN,NaN,NaN
73,LMC,Lower middle income,NaN,NaN,NaN
81,MNA,"Middle East, North Africa, Afghanistan & Pakis...",NaN,NaN,NaN


In [28]:
country_meta_clean = country_meta_clean.dropna(subset=["Region"])
country_meta_clean.shape

(120, 5)

In [29]:
country_meta_clean.isna().sum()

Country Code        0
Country Name        0
Region              0
Income Group        1
Lending Category    0
dtype: int64

In [30]:
country_meta_clean[country_meta_clean["Income Group"].isna()]

,Country Code,Country Name,Region,Income Group,Lending Category
39,ETH,Ethiopia,Sub-Saharan Africa,NaN,IDA


In [31]:
country_meta_clean["Income Group"] = country_meta_clean["Income Group"].fillna("Not Classified")
country_meta_clean.isna().sum()

Country Code        0
Country Name        0
Region              0
Income Group        0
Lending Category    0
dtype: int64

In [32]:
main_country_codes = set(df_long["Country Code"].unique())
aggregate_codes = set(country_meta[country_meta["Region"].isna()]["Code"].unique())

overlap = main_country_codes.intersection(aggregate_codes)
print(len(overlap))
print(overlap)

0
set()


In [33]:
df_long["Country Code"].nunique()

134

In [76]:
country_meta_clean["Country Code"] = country_meta_clean["Country Code"].str.strip()

In [77]:
country_meta_clean.to_csv("../data/cleaned/country_meta_cleaned.csv", index=False)

In [35]:
print(series_meta.columns.tolist())
series_meta.head()

['Code', 'License Type', 'Indicator Name', 'Short definition', 'Long definition', 'Source', 'Topic', 'Dataset', 'Periodicity', 'Aggregation method', 'Limitations and exceptions', 'General comments']


,Code,License Type,Indicator Name,Short definition,Long definition,Source,Topic,Dataset,Periodicity,Aggregation method,Limitations and exceptions,General comments
0,DT.GPA.DPPG,NaN,Average grace period on new external debt comm...,Grace period is the period from the date of si...,Grace period is the period from the date of si...,"World Bank, International Debt Statistics.",Economic Policy & Debt: External debt: Terms,International Debt Statistics,Annual,Weighted average,NaN,NaN
1,DT.GPA.OFFT,NaN,Average grace period on new external debt comm...,Grace period is the period from the date of si...,Grace period is the period from the date of si...,"World Bank, International Debt Statistics.",Economic Policy & Debt: External debt: Terms,International Debt Statistics,Annual,Weighted average,NaN,NaN
2,DT.GPA.PRVT,NaN,Average grace period on new external debt comm...,Grace period is the period from the date of si...,Grace period is the period from the date of si...,"World Bank, International Debt Statistics.",Economic Policy & Debt: External debt: Terms,International Debt Statistics,Annual,Weighted average,NaN,NaN
3,DT.GRE.DPPG,NaN,Average grant element on new external debt com...,The grant element of a loan is the grant equiv...,The grant element of a loan is the grant equiv...,"World Bank, International Debt Statistics.",Economic Policy & Debt: External debt: Terms,International Debt Statistics,Annual,Weighted average,NaN,NaN
4,DT.GRE.OFFT,NaN,Average grant element on new external debt com...,The grant element of a loan is the grant equiv...,The grant element of a loan is the grant equiv...,"World Bank, International Debt Statistics.",Economic Policy & Debt: External debt: Terms,International Debt Statistics,Annual,Weighted average,NaN,NaN


In [36]:
series_meta_clean = series_meta[["Code", "Indicator Name", "Topic"]].copy()
series_meta_clean.columns = ["Series Code", "Indicator Name", "Topic"]
series_meta_clean.head()

,Series Code,Indicator Name,Topic
0,DT.GPA.DPPG,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms
1,DT.GPA.OFFT,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms
2,DT.GPA.PRVT,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms
3,DT.GRE.DPPG,Average grant element on new external debt com...,Economic Policy & Debt: External debt: Terms
4,DT.GRE.OFFT,Average grant element on new external debt com...,Economic Policy & Debt: External debt: Terms


In [37]:
series_meta_clean.isna().sum()


Series Code       0
Indicator Name    0
Topic             0
dtype: int64

In [38]:
series_meta_clean.duplicated().sum()

np.int64(0)

In [39]:
main_series_codes = set(df_long["Series Code"].unique())
meta_series_codes = set(series_meta_clean["Series Code"].unique())

print("In main data but not in metadata:", len(main_series_codes - meta_series_codes))
print("In metadata but not in main data:", len(meta_series_codes - main_series_codes))
print("Matching codes:", len(main_series_codes.intersection(meta_series_codes)))

In main data but not in metadata: 73
In metadata but not in main data: 71
Matching codes: 503


In [40]:
missing_codes = main_series_codes - meta_series_codes
df_long[df_long["Series Code"].isin(missing_codes)][["Series Code", "Series Name"]].drop_duplicates().head(20)

,Series Code,Series Name
295,DT.DOD.DIMF.US.CD,"Use of IMF credit (DOD, current US$)"
106,DT.INT.DIMF.US.CD,"IMF credit, charges (INT, current US$)"
111,DT.INT.DSDR.CD,"IMF SDR allocations, charges (INT, current US$)"
309,DT.AMT.BLAT.CB.CD,"CB, bilateral (AMT, current US$)"
316,DT.AMT.BLTC.CB.CD,"CB, bilateral concessional (AMT, current US$)"
698,DT.AMT.DECB.CD,"Principal repayments on external debt, central..."
323,DT.AMT.MLAT.CB.CD,"CB, multilateral (AMT, current US$)"
330,DT.AMT.MLTC.CB.CD,"CB, multilateral concessional (AMT, current US$)"
337,DT.AMT.OFFT.CB.CD,"CB, official creditors (AMT, current US$)"
310,DT.DIS.BLAT.CB.CD,"CB, bilateral (DIS, current US$)"


In [41]:
series_meta_clean = series_meta[["Code", "Topic"]].copy()
series_meta_clean.columns = ["Series Code", "Topic"]
series_meta_clean.head()

,Series Code,Topic
0,DT.GPA.DPPG,Economic Policy & Debt: External debt: Terms
1,DT.GPA.OFFT,Economic Policy & Debt: External debt: Terms
2,DT.GPA.PRVT,Economic Policy & Debt: External debt: Terms
3,DT.GRE.DPPG,Economic Policy & Debt: External debt: Terms
4,DT.GRE.OFFT,Economic Policy & Debt: External debt: Terms


In [78]:
series_meta_clean["Series Code"] = series_meta_clean["Series Code"].str.strip()

In [79]:
series_meta_clean.to_csv("../data/cleaned/series_meta_cleaned.csv", index=False)

In [43]:
print(footnote_meta.columns.tolist())
footnote_meta.head()

['Type', 'Country Code', 'Series Code', 'Time Code', 'Description']


,Type,Country Code,Series Code,Time Code,Description
0,FootNote,Afghanistan (AFG),Personal transfers and compensation of employe...,2024 (YR2024),Data on Personal Transfers and Compensation of...
1,FootNote,Afghanistan (AFG),Personal transfers and compensation of employe...,2023 (YR2023),Source: United Nations Conference on Trade and...
2,FootNote,Afghanistan (AFG),Personal transfers and compensation of employe...,2021 (YR2021),Source: United Nations Conference on Trade and...
3,FootNote,Afghanistan (AFG),Personal transfers and compensation of employe...,2022 (YR2022),Source: United Nations Conference on Trade and...
4,FootNote,Afghanistan (AFG),"Foreign direct investment, net inflows in repo...",2021 (YR2021),Source: United Nations Conference on Trade and...


In [44]:
print(country_series.columns.tolist())
country_series.head()

['Type', 'Country Code', 'Series Code', 'Description']


,Type,Country Code,Series Code,Description
0,Country-Series,Afghanistan (AFG),"Foreign direct investment, net inflows in repo...",Data on Foreign Direct Investment are based on...
1,Country-Series,Afghanistan (AFG),"Population, total (SP.POP.TOTL)",Data source: United Nations World Population P...
2,Country-Series,Afghanistan (AFG),"External debt stocks, total (DOD, current US$)...","Long-term public and publicly guaranteed, long..."
3,Country-Series,Angola (AGO),"External debt stocks, total (DOD, current US$)...",Long-term public and publicly guaranteed debt ...
4,Country-Series,Angola (AGO),"Population, total (SP.POP.TOTL)",Data source: United Nations World Population P...


In [45]:
df_long["Value"].describe()

count    1.487450e+06
mean     8.578855e+09
std      1.977765e+11
min     -5.270000e+11
25%      0.000000e+00
50%      8.634973e+06
75%      2.876893e+08
max      3.820000e+13
Name: Value, dtype: float64

In [46]:
df_long.groupby("Series Name")["Value"].describe().sort_values("max", ascending=False).head(10)

,count,mean,std,min,25%,50%,75%,max
Series Name,,,,,,,,
GNI (current US$),3350.0,7.883110e+11,3.483387e+12,6.994144e+07,4.858283e+09,1.717451e+10,1.150000e+11,3.820000e+13
"Imports of goods, services and primary income (current US$)",3325.0,2.164916e+11,9.102223e+11,4.122914e+07,2.499504e+09,8.265381e+09,4.466830e+10,1.030000e+13
"Exports of goods, services and primary income (current US$)",3325.0,2.031429e+11,8.825349e+11,1.236564e+07,1.548912e+09,6.059926e+09,3.775680e+10,9.910000e+12
"External debt stocks, total (DOD, current US$)",3350.0,1.962375e+11,8.087238e+11,6.958315e+07,1.959931e+09,7.726527e+09,4.053476e+10,8.940000e+12
"Total reserves (includes gold, current US$)",3275.0,1.697600e+11,7.653802e+11,2.677074e+05,5.454445e+08,2.552649e+09,2.248140e+10,6.530000e+12
"External debt stocks, long-term (DOD, current US$)",3350.0,1.411649e+11,5.598315e+11,1.000000e+04,1.460383e+09,6.416359e+09,3.310453e+10,6.170000e+12
"External debt stocks, variable rate (DOD, current US$)",3300.0,8.687024e+10,3.603535e+11,0.000000e+00,1.130572e+08,2.192483e+09,1.655691e+10,3.780000e+12
"External debt stocks, public and publicly guaranteed (PPG) (DOD, current US$)",3350.0,7.962774e+10,3.002585e+11,1.000000e+04,1.267831e+09,4.455420e+09,2.246024e+10,3.560000e+12
"External debt stocks, long-term public sector (DOD, current US$)",3350.0,7.937990e+10,2.991143e+11,1.000000e+04,1.264751e+09,4.455420e+09,2.245762e+10,3.530000e+12


In [47]:
series_meta_clean["Topic"].value_counts()

Topic
Economic Policy & Debt: External debt: Debt outstanding                                   77
Economic Policy & Debt: External debt: Net flows                                          75
Economic Policy & Debt: External debt: Interest                                           71
Economic Policy & Debt: External debt: Amortization                                       68
Economic Policy & Debt: External debt: Disbursements                                      68
Economic Policy & Debt: External debt: Debt service                                       68
Economic Policy & Debt: External debt: Net transfers                                      67
Economic Policy & Debt: External debt: Arrears, reschedulings, etc.                       22
Economic Policy & Debt: External debt: Terms                                              12
Economic Policy & Debt: External debt: Currency composition                               10
Economic Policy & Debt: External debt: Debt ratios & other items

In [48]:
series_meta[series_meta["Topic"] == "Health: Population: Structure"]

,Code,License Type,Indicator Name,Short definition,Long definition,Source,Topic,Dataset,Periodicity,Aggregation method,Limitations and exceptions,General comments
340,SP.POP.TOTL,CC BY-4.0,"Population, total",NaN,Total population is based on the de facto defi...,(1) United Nations Population Division. World ...,Health: Population: Structure,NaN,Annual,Sum,Current population estimates for developing co...,Relevance to gender indicator: disaggregating ...


In [49]:
indicator_coverage = df_long.groupby("Series Name")["Value"].count().sort_values(ascending=False)
indicator_coverage.describe()

count     576.000000
mean     2582.378472
std       962.598288
min       200.000000
25%      1900.000000
50%      3075.000000
75%      3350.000000
max      3350.000000
Name: Value, dtype: float64

In [50]:
df_long["Series Name"].nunique()


576

In [51]:
df_long["Series Code"].nunique()

576

In [81]:
import pandas as pd

df_long = pd.read_csv("../data/cleaned/All_country_data_cleaned.csv")
print("Before strip:", repr(df_long["Country Code"].iloc[0]))

df_long["Country Code"] = df_long["Country Code"].str.strip()
df_long["Series Code"] = df_long["Series Code"].str.strip()

print("After strip:", repr(df_long["Country Code"].iloc[0]))

Before strip: 'AFG       '
After strip: 'AFG'


In [82]:
check = pd.read_csv("../data/cleaned/All_country_data_cleaned.csv")
print(repr(check["Country Code"].iloc[0]))

'AFG       '


In [84]:
import pandas as pd

# Load fresh
df_long = pd.read_csv("../data/cleaned/All_country_data_cleaned.csv")
print("Loaded. Before strip:", repr(df_long["Country Code"].iloc[0]))

# Strip
df_long["Country Code"] = df_long["Country Code"].astype(str).str.strip()
df_long["Series Code"] = df_long["Series Code"].astype(str).str.strip()
print("After strip:", repr(df_long["Country Code"].iloc[0]))

# Save
df_long.to_csv("../data/cleaned/All_country_data_cleaned.csv", index=False)
print("Saved.")

# Re-read from disk to verify
check = pd.read_csv("../data/cleaned/All_country_data_cleaned.csv")
print("Verified from disk:", repr(check["Country Code"].iloc[0]))

Loaded. Before strip: 'AFG'
After strip: 'AFG'
Saved.
Verified from disk: 'AFG'


In [86]:
# Country meta
country_meta_clean = pd.read_csv("../data/cleaned/country_meta_cleaned.csv")
print("Before:", repr(country_meta_clean["Country Code"].iloc[0]))
country_meta_clean["Country Code"] = country_meta_clean["Country Code"].astype(str).str.strip()
country_meta_clean.to_csv("../data/cleaned/country_meta_cleaned.csv", index=False)
check1 = pd.read_csv("../data/cleaned/country_meta_cleaned.csv")
print("Verified:", repr(check1["Country Code"].iloc[0]))

# Series meta
series_meta_clean = pd.read_csv("../data/cleaned/series_meta_cleaned.csv")
print("Before:", repr(series_meta_clean["Series Code"].iloc[0]))
series_meta_clean["Series Code"] = series_meta_clean["Series Code"].astype(str).str.strip()
series_meta_clean.to_csv("../data/cleaned/series_meta_cleaned.csv", index=False)
check2 = pd.read_csv("../data/cleaned/series_meta_cleaned.csv")
print("Verified:", repr(check2["Series Code"].iloc[0]))

Before: 'AFG'
Verified: 'AFG'
Before: 'DT.GPA.DPPG'
Verified: 'DT.GPA.DPPG'


In [88]:
import pandas as pd

df_long = pd.read_csv("../data/cleaned/All_country_data_cleaned.csv")
country_meta_clean = pd.read_csv("../data/cleaned/country_meta_cleaned.csv")

main_codes = set(df_long["Country Code"].unique())
country_table_codes = set(country_meta_clean["Country Code"].unique())

missing_in_countries = main_codes - country_table_codes

print("Total unique codes in debt data:", len(main_codes))
print("Total codes in countries table:", len(country_table_codes))
print("Codes in debt data but MISSING from countries table:", len(missing_in_countries))
print(sorted(missing_in_countries))

Total unique codes in debt data: 134
Total codes in countries table: 120
Codes in debt data but MISSING from countries table: 14
['EAP', 'ECA', 'IDA', 'IDX', 'LAC', 'LDC', 'LIC', 'LMC', 'LMY', 'MIC', 'MNA', 'SAS', 'SSA', 'UMC']


In [89]:
aggregate_codes = ['EAP', 'ECA', 'IDA', 'IDX', 'LAC', 'LDC', 'LIC', 'LMC', 'LMY', 'MIC', 'MNA', 'SAS', 'SSA', 'UMC']

print("Before removing aggregates:", df_long.shape)
df_long = df_long[~df_long["Country Code"].isin(aggregate_codes)]
print("After removing aggregates:", df_long.shape)

print("Unique countries now:", df_long["Country Code"].nunique())

# Save the corrected file
df_long.to_csv("../data/cleaned/All_country_data_cleaned.csv", index=False)

# Verify from disk
check = pd.read_csv("../data/cleaned/All_country_data_cleaned.csv")
print("Verified unique countries on disk:", check["Country Code"].nunique())

Before removing aggregates: (1487450, 8)
After removing aggregates: (1290500, 8)
Unique countries now: 120
Verified unique countries on disk: 120
